In [1]:
from google.colab import userdata
import os
os.environ["LANGCHAIN_TRACING"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = userdata.get('langsmith')
os.environ["LANGCHAIN_PROJECT"] = "Managing Long Conversations"

In [2]:
!pip install -U langchain-google-genai -q
from langchain_google_genai import ChatGoogleGenerativeAI


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.8 MB/s eta 0:00:00


In [4]:
API_key = userdata.get('gemini')

llm_model = "gemini-2.5-flash"
os.environ["GOOGLE_API_KEY"] =  API_key
model = ChatGoogleGenerativeAI(
    model=llm_model,
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)


# Summarize messages

In [6]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [7]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to gather factual information about the fictional moon capital, Lunapolis, including its basic details, weather, population demographics (specifically cheese miners), and potential social events (a strike by the cheese miners' union).\n\n## SUMMARY\nThe conversation has established the following key facts about Lunapolis:\n*   Lunapolis is the capital of the moon.\n*   The weather in Lunapolis is clear, with a high of 120C and a low of -100C.\n*   There are 100,000 cheese miners residing in Lunapolis.\n*   The cheese miners' union is expected to strike due to their dissatisfaction with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nAwait further questions or instructions from the user regarding Lunapolis or related topics.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='ede61e0d-9935-4eed-a72e-d0c07b257d93'),
     

In [8]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user's primary goal is to gather factual information about the fictional moon capital, Lunapolis, including its basic details, weather, population demographics (specifically cheese miners), and potential social events (a strike by the cheese miners' union).

## SUMMARY
The conversation has established the following key facts about Lunapolis:
*   Lunapolis is the capital of the moon.
*   The weather in Lunapolis is clear, with a high of 120C and a low of -100C.
*   There are 100,000 cheese miners residing in Lunapolis.
*   The cheese miners' union is expected to strike due to their dissatisfaction with the new president.

## ARTIFACTS
None

## NEXT STEPS
Await further questions or instructions from the user regarding Lunapolis or related topics.


# Trim/delete messages

In [9]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage


In [10]:
@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]

    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [11]:
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [12]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='0674cb33-43ee-4e67-b8c8-a7856a0c02d0'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='7d8c6718-8ebf-4838-ab4a-98e4f7e39159', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='535158ba-b865-4a7e-8cea-55883321b4ba'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='992ce302-db52-4492-83d2-de6a05aee6c2', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='bf216145-a728-4f47-aac2-408702d4853f'),
              AIMessage(content="I can't tell you the exact temperature of your device, as I do

In [13]:
print(response["messages"][-1].content)

I can't tell you the exact temperature of your device, as I don't have a way to sense it directly.

However, its temperature can be a very important clue!

*   **Is the device unusually hot to the touch?** This could indicate overheating, which often causes devices to shut down or prevent them from turning on as a safety measure.
*   **Is it unusually cold?** Some devices, especially batteries, don't function well in extreme cold.
*   **Does it feel normal (room temperature)?**

Please feel the device and let me know if it feels hot, cold, or normal.
